In [1]:
import mne
import tensorly as tl
import os

backend = 'numpy'
os.environ['TENSORLY_BACKEND'] = backend
tl.set_backend(backend)
mne.set_log_level('ERROR')

print(mne.get_config_path())
print(mne.get_config('MNE_DATA'))
print(mne.get_config('MOABB_RESULTS'))
print(tl.get_backend())
%pip freeze | grep moabb

/user/leuven/352/vsc35289/.mne/mne-python.json
/scratch/leuven/352/vsc35289/mne_data
/data/leuven/352/vsc35289/moabb_results
numpy
moabb==1.1.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
%env TENSORLY_BACKEND

'numpy'

In [3]:
import multiprocessing

multiprocessing.cpu_count()

36

In [4]:
from moabb.paradigms import P300
from moabb.datasets import *
from moabb.evaluations import WithinSessionEvaluation

paradigm = P300(resample=48)

datasets = [
    BNCI2014_008(),
    #BNCI2015_003(),
    #BNCI2014_009(),
    #BI2012(),
    #BI2013a(),                               
    #BI2014a(),
    #DemonsP300(),
    #Cattan2019_VR(),
    #EPFLP300(),
    #BI2015a(),   
    #BI2015b(),
    #BI2014b(),
    #Lee2019_ERP(),
    #Huebner2017(),
    #Huebner2018(),
    #Sosulski2019(),
]



In [5]:
from sklearn.pipeline import make_pipeline, Pipeline
from hoda.hoda import HODA, BTTDA,  GreedyBTTDA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from hoda.classification import ToeplitzLDAWrapper,ZScore
from sklearn.preprocessing import StandardScaler
from mne.decoding import Scaler
from sklearn.linear_model import LogisticRegression
from hoda.tensorize import  Vectorize, Tensorize
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import StratifiedKFold
from hoda.classification import SelectF

pipelines=dict()

cv=StratifiedKFold(random_state=42, shuffle=True)


hoda_params = dict(
    max_iter=64,
    tol=1e-8,
    init ='random',
    random_state=42,
    shrinkage='lw',
    solver='lanczos',
    taper=False,
    forward=False,
    obj='tr',
    toeplitz=None,
    delta=None,
    extra_train_info=False,
)

bttda_params=dict(
        truncate=True,
        n_jobs=1,
        hoda_params=hoda_params,
        extra_train_info=False,
        verbose=False,
        cv=cv,
)


clf = make_pipeline(
    SelectF(alpha=.05),
    FunctionTransformer(tl.to_numpy),
    StandardScaler(),
    LDA(shrinkage='auto', solver='lsqr')
)
pipelines['BTTDA_10'] = Pipeline([
    ('zscore1', ZScore()),
    ('bttda', GreedyBTTDA(
        max_blocks=10,
        **bttda_params,
        clf=clf,
    )),
    ('clf', clf)
])


pipelines['HODA'] = Pipeline([
    ('zscore1', ZScore()),
    ('bttda', GreedyBTTDA(
        max_blocks=1,
        **bttda_params,
        clf=clf,
    )),
    ('clf', clf)
])

pipelines['PARAFACDA_10'] = Pipeline([
    ('zscore1', ZScore()),
    ('bttda', GreedyBTTDA(
        max_blocks=10,
        **bttda_params,
        clf=clf,
        rank_grid=[1],
    )),
    ('clf', clf)
])


In [6]:
import pandas as pd

results = []
for dataset in datasets:
    print(dataset.code)
    evaluation = WithinSessionEvaluation(
        paradigm=paradigm,
        datasets=[dataset],
        overwrite=False,
        random_state=42,
        n_jobs=5,
        suffix=f'bttda_rev2_{dataset.code}',
    )
    results.append(evaluation.process(pipelines, postprocess_pipeline=FunctionTransformer(tl.tensor)))
results = pd.concat(results)

BNCI2014-008


BNCI2014-008-WithinSession:   0%|          | 0/8 [00:00<?, ?it/s]/data/leuven/352/vsc35289/miniconda3/envs/bttda/lib/python3.12/site-packages/moabb/datasets/preprocessing.py:279: UserWarning: warnEpochs <Epochs |  4200 events (all good), 0 – 1 s, baseline off, ~65.9 MB, data loaded,
 'Target': 700
 'NonTarget': 3500>
  warn(f"warnEpochs {epochs}")


No hdf5_path provided, models will not be saved.
No hdf5_path provided, models will not be saved.
No hdf5_path provided, models will not be saved.


BNCI2014-008-WithinSession:  12%|█▎        | 1/8 [07:56<55:34, 476.31s/it]/data/leuven/352/vsc35289/miniconda3/envs/bttda/lib/python3.12/site-packages/moabb/datasets/preprocessing.py:279: UserWarning: warnEpochs <Epochs |  4200 events (all good), 0 – 1 s, baseline off, ~65.9 MB, data loaded,
 'Target': 700
 'NonTarget': 3500>
  warn(f"warnEpochs {epochs}")


No hdf5_path provided, models will not be saved.
No hdf5_path provided, models will not be saved.
No hdf5_path provided, models will not be saved.


...

BNCI2014-008-WithinSession:  38%|███▊      | 3/8 [25:27<40:57, 491.42s/it]  /data/leuven/352/vsc35289/miniconda3/envs/bttda/lib/python3.12/site-packages/moabb/datasets/preprocessing.py:279: UserWarning: warnEpochs <Epochs |  4200 events (all good), 0 – 1 s, baseline off, ~65.9 MB, data loaded,
 'Target': 700
 'NonTarget': 3500>
  warn(f"warnEpochs {epochs}")


No hdf5_path provided, models will not be saved.
No hdf5_path provided, models will not be saved.
No hdf5_path provided, models will not be saved.

Selected rank (8, 32) with score 0.8161
New ranks: [(8, 32)]



Selected rank (1, 1) with score 0.8203
New ranks: [(8, 32), (1, 1)]



Selected rank (8, 32) with score 0.8211
New ranks: [(8, 32), (1, 1), (8, 32)]



Selected rank (8, 32) with score 0.8218
New ranks: [(8, 32), (1, 1), (8, 32), (8, 32)]



Selected rank (8, 16) with score 0.8221
New ranks: [(8, 32), (1, 1), (8, 32), (8, 32), (8, 16)]



Selected rank (4, 4) with score 0.8225
New ranks: [(8, 32), (1, 1), (8, 32), (8, 32), (8, 16), (4, 4)]



Selected rank (2, 2) with score 0.8221
New ranks: [(8, 32), (1, 1), (8, 32), (8, 32), (8, 16), (4, 4), (2, 2)]



Selected rank (1, 1) with score 0.8224
New ranks: [(8, 32), (1, 1), (8, 32), (8, 32), (8, 16), (4, 4), (2, 2), (1, 1)]



Selected rank (8, 8) with score 0.8242
New ranks: [(8, 32), (1, 1), (8, 32), (8, 32), (8, 16), (4, 4), (2

BNCI2014-008-WithinSession:  50%|█████     | 4/8 [32:06<30:19, 454.82s/it]/data/leuven/352/vsc35289/miniconda3/envs/bttda/lib/python3.12/site-packages/moabb/datasets/preprocessing.py:279: UserWarning: warnEpochs <Epochs |  4200 events (all good), 0 – 1 s, baseline off, ~65.9 MB, data loaded,
 'Target': 700
 'NonTarget': 3500>
  warn(f"warnEpochs {epochs}")


No hdf5_path provided, models will not be saved.
No hdf5_path provided, models will not be saved.
No hdf5_path provided, models will not be saved.


BNCI2014-008-WithinSession:  50%|█████     | 4/8 [36:35<36:35, 548.79s/it]

KeyboardInterrupt



In [ ]:
results.to_csv('moabb_results_erp.csv')

In [ ]:
results

In [ ]:
(results.groupby(['dataset', 'pipeline']).score.aggregate(['mean', 'std'])*100 ).round(2)

##### results

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

order = results.groupby('pipeline')
order = order.score.aggregate('mean')
order = order.sort_values()


sns.catplot(data=results , x='session', y='score', col='dataset',hue='pipeline', col_wrap=3,kind='bar')
plt.show()

In [ ]:
sns.barplot(data=results, x='pipeline', y='score')

In [ ]:
from moabb.analysis.meta_analysis import compute_dataset_statistics, find_significant_differences
from moabb.analysis.plotting import summary_plot
import matplotlib.pyplot as plt

stats = compute_dataset_statistics(results)
P, T = find_significant_differences(stats)
_ = summary_plot(P, T, simplify=False)
plt.show()

In [ ]:
from moabb.analysis.plotting import paired_plot
_ = paired_plot(results, 'PARAFACDA_10', 'BTTDA_10')

In [ ]:
import moabb.analysis.plotting as moabb_plt

_ = moabb_plt.meta_analysis_plot(stats, 'BTTDA_10', 'PARAFACDA_10')

In [ ]:
results[results['dataset'] == 'BrainInvaders2015b'].groupby(['session', 'subject']).aggregate('count')